In [ ]:
#Frequency-only (FCT /DST)
import os, cv2, numpy as np, random
import matplotlib.pyplot as plt
import sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report
import tensorflow as tf
from scipy.fft import dst
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FCT(DST) shape: (9658, 128, 128, 1) labels: (9658,)
Epoch 1/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 67s 296ms/step - accuracy: 0.6258 - loss: 0.6710 - val_accuracy: 0.9141 - val_loss: 0.2148
Epoch 2/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 79s 281ms/step - accuracy: 0.9289 - loss: 0.1796 - val_accuracy: 0.9788 - val_loss: 0.0713
Epoch 3/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 83s 287ms/step - accuracy: 0.9668 - loss: 0.0867 - val_accuracy: 0.9840 - val_loss: 0.0577
Epoch 4/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 82s 286ms/step - accuracy: 0.9826 - loss: 0.0511 - val_accuracy: 0.9720 - val_loss: 0.0759
Epoch 5/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 82s 286ms/step - accuracy: 0.9854 - loss: 0.0463 - val_accuracy: 0.9928 - val_loss: 0.0290
Epoch 6/10
221/221 ━━━━━━━━━━━━━━━━━━━━ 82s 288ms/step - accuracy: 0.9878 - loss: 0.0375 - val_accuracy: 0.9928 - val_loss: 0.0271
Epoch 7/10
221/221 ━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SEED=15
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
REAL_DIR = "/content/drive/My Drive/Research Project/ffhq_7k/train"
FAKE_DIR = "/content/drive/My Drive/Research Project/StyleGAN2_7k/train"
IMG_SIZE=(128,128)

In [ ]:
def load_fct(real_dir, fake_dir, img_size=(128,128), limit=None):
    X, y = [], []
    def proc(p):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, img_size).astype(np.float32)/255.0

        f = dst(dst(img, type=2, norm='ortho', axis=0), type=2, norm='ortho', axis=1)
        f = np.log(np.abs(f)+1e-8)

        return f[..., None]
    reals = sorted(os.listdir(real_dir))
    fakes = sorted(os.listdir(fake_dir))
    if limit: reals, fakes = reals[:limit], fakes[:limit]
    for fn in reals: X.append(proc(os.path.join(real_dir, fn))); y.append(0)
    for fn in fakes: X.append(proc(os.path.join(fake_dir, fn))); y.append(1)
    return np.array(X), np.array(y)

In [ ]:
X, y = load_fct(REAL_DIR, FAKE_DIR)
print("FCT(DST) shape:", X.shape, "labels:", y.shape)

In [ ]:
def build_branch(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(16,(4,4),activation='relu')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32,(3,3),activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64,activation='relu')(x)
    out = layers.Dense(1,activation='sigmoid')(x)
    return models.Model(inp,out)

In [ ]:
model = build_branch((128,128,1))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
early_stop = EarlyStopping(monitor='val_loss', patience=9, restore_best_weights=True)
history = model.fit(Xtr, ytr, validation_data=(Xte,yte), epochs=20, batch_size=35, callbacks=[early_stop], verbose=1)

print("Test eval:", model.evaluate(Xte, yte, verbose=0))

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

from google.colab import drive
drive.mount('/content/drive')

REAL_TEST_DIR = "/content/drive/My Drive/Research Project/ffhq_2k"
FAKE_TEST_DIR = "/content/drive/My Drive/Research Project/Stylegan3_2k"

X_test,  y_test = load_fct(REAL_TEST_DIR, FAKE_TEST_DIR, img_size=IMG_SIZE, limit=None)

print("StyleGAN3 Test Shapes:", X_test.shape, y_test.shape)


test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print("StyleGAN3 Test Loss:", test_loss)
print("StyleGAN3 Test Accuracy:", test_acc)


y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

print("\nClassification Report (StyleGAN3 Test Set):")
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
StyleGAN3 Test Shapes: (4079, 128, 128, 1) (4079,)
128/128 ━━━━━━━━━━━━━━━━━━━━ 8s 63ms/step - accuracy: 0.9944 - loss: 0.0259
StyleGAN3 Test Loss: 0.01900722086429596
StyleGAN3 Test Accuracy: 0.9960774779319763
128/128 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step

Classification Report (StyleGAN3 Test Set):
              precision    recall  f1-score   support

        Real       1.00      0.99      1.00      2079
        Fake       0.99      1.00      1.00      2000

    accuracy                           1.00      4079
   macro avg       1.00      1.00      1.00      4079
weighted avg       1.00      1.00      1.00      4079

